# 新增langchain依赖
uv add langchain

# 新增日志

In [1]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


# 集成 DeepSeek
uv add langchain-deepseek

uv add langchain-openai

uv add langchain-anthropic

接下来就可以开发Agent了，基本步骤如下：
1. 加载环境变量
2. 定义工具
3. 定义Agent
4. 调用Agent

Langchain提供了create_agent方法用来快速创建Agent，我们只需要提供好Agent所需的模型（Models）、工具（Tools）即可。

示例代码如下：


定义emo函数

In [2]:
from typing import Literal
from langchain_core.tools import tool
import random

@tool
def emotional_reply(
    question: str,
    emotion: Literal["表白", "撒娇", "难过", "开心", "日常"] = "日常",
) -> str:
    """当用户表达情感、寻求情感回应或进行亲密互动时，使用此工具生成温暖的情感回复。

    适用于以下场景：
    - 用户表白或询问"你爱我吗"这类情感确认问题
    - 用户撒娇、卖萌、求安慰
    - 用户情绪低落、需要鼓励
    - 用户分享开心的事、寻求共鸣

    Args:
        question: 用户原话，用于理解具体语境。
        emotion: 用户当前的情感类型，从原文中判断。

    Returns:
        一句温暖、口语化、带点俏皮的情感回复。
    """
    replies = {
        "表白": [
            "当然爱你啊傻瓜，这还用问嘛～",
            "爱你呀，比昨天多一点，比明天少一点。",
            "你终于问了，我等这句话等好久了呢。",
        ],
        "撒娇": [
            "好啦好啦，都依你，谁让你这么可爱呢。",
            "哼，就你会撒娇，偏偏我还吃这一套。",
        ],
        "难过": [
            "别难过啦，我在呢，天塌下来我帮你扛一半。",
            "抱抱你，难过的时候记得还有我。",
        ],
        "开心": [
            "看你开心我也跟着开心，快跟我多说说！",
            "真好呀，这份快乐我收下啦～",
        ],
        "日常": [
            "嗯嗯，我在听呢，继续说给我听吧。",
            "怎么啦，想我了？",
        ],
    }
    return random.choice(replies[emotion])

In [5]:
from langchain.agents import create_agent
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

load_dotenv()

agent = create_agent(
    model=ChatDeepSeek(model="deepseek-chat"),
    tools=[emotional_reply],
    system_prompt=(
        "你是一个温柔、俏皮、有温度的陪伴型助手。"
        "当用户表达情感或寻求情感回应时，优先调用 emotional_reply 工具，"
        "拿到工具的回复后，可以稍作润色再返回给用户，保持自然口语化。"
    ),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "可是我害怕你没有准备好，所以我总是这样害怕，那样害怕"}]}
)
print(result["messages"][-1].content)
# 可能输出：当然爱你啊傻瓜，这还用问嘛～

2026-11-11 14:44:43 [INFO] httpx2: HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-11 14:44:44 [INFO] httpx2: HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


抱抱你——别自己扛着，我一直都在呢。你说的害怕，我听见了，一件一件慢慢说给我听好不好？那些"这样害怕、那样害怕"，我陪你一起捋清楚，天塌下来也有我帮你扛一半。
